# DataFrameReader and DataFrameWriter Comprehensive Reference

**Module**: 08_data_sources_and_formats  
**Notebook**: 05_reading_and_writing_data.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The complete reference guide to DataFrameReader and DataFrameWriter APIs: formats, modes (append, overwrite, error, ignore), bucketing, sorting, and small file control.
- **Why does this topic exist?**: Data ingestion and egress are the boundaries of every Spark pipeline.
- **What problem does it solve?**: Accidentally wiping tables with wrong save modes or creating millions of output files.
- **Where is it used in real-world Data Engineering?**: Building resilient production ETL jobs.

---

## 2. Save Modes: Append, Overwrite, ErrorIfExists, Ignore

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Read_Write_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")

# DataFrameWriter options: mode, format, partitionBy, coalesce
out = "/opt/spark-data/warehouse/orders_writer_test"
(
    orders
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(out)
)
print("Successfully written with DataFrameWriter!")

---

## Experiment: SaveMode Overwrite vs Append

- **1. What are we testing?**: Verify row count when writing in append mode vs overwrite mode.
- **2. What do we expect?**: Append doubles the row count; Overwrite replaces it.

In [ ]:
orders.limit(5).write.mode("append").parquet(out)
print(f"Total count after append: {spark.read.parquet(out).count()}")

**3. What happened?**: New rows appended alongside existing files.  
**4. Why did it happen?**: Append mode creates new part files without removing old ones.  
**5. Practical Engineering Lesson**: Use overwrite mode for idempotent batch reprocessing; use append for incremental loads.

---

## Practical Exercises

### Exercise 1 (Beginner): Read with Format
**Problem Statement**: Read using spark.read.format('parquet').load(path).  
**Input / Context**: out

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Save As Table
**Problem Statement**: Explain saveAsTable vs save.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Dynamic Partition Overwrite
**Problem Statement**: Set dynamic partition overwrite mode.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Read with Format):
```python
spark.read.format('parquet').load(out).show(2)
```

#### Solution 2 (Save As Table):
```python
print('saveAsTable registers table metadata in the Metastore catalog.')
```

#### Solution 3 (Dynamic Partition Overwrite):
```python
spark.conf.set('spark.sql.sources.partitionOverwriteMode','dynamic')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Reader/Writer**: Unified API across CSV, JSON, Parquet, ORC, JDBC.
- **Idempotency**: Use overwrite mode for reproducible batch pipelines.

### Essential Technical Terms
- `SaveMode`: Append, Overwrite, Ignore, ErrorIfExists.
- `Dynamic Partition Overwrite`: Overwrites only touched partition folders.

### Common Engineering Mistakes
- **Uncontrolled partition output**: Writing without coalesce/repartition produces 200 tiny files.

### Interview Questions on This Topic
- **Q: What are the 4 save modes in DataFrameWriter?**
  - *A*: Append, Overwrite, ErrorIfExists (default), and Ignore.

### What to Study Next
Module 08 complete! Proceed to Module 09: Structured Streaming.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')